# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hanizakkk/flyrank_working-repo/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [45]:
import os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF token loaded:", bool(HF_TOKEN))

HF token loaded: True


In [46]:
!pip -q install duckdb fsspec huggingface_hub

In [47]:
import duckdb

con = duckdb.connect()

print("DuckDB connected successfully.")

DuckDB connected successfully.


In [48]:
con.execute(f"""
INSTALL httpfs;
LOAD httpfs;

CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

print("Hugging Face connection configured.")

Hugging Face connection configured.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
Unit of analysis: One row represents the daily search and analytics performance of one content item for one client on one report date.

Time window: I will use March 2026 as the development window. The warehouse covers 2025-01-27 through 2026-06-30, but June 2026 is treated as a sealed final month and is not used for developing the label logic.

In [49]:
con.execute("""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY client_hash_id, content_hash_id, report_date
HAVING COUNT(*) > 1
LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,row_count


In [50]:
df_march = con.execute("""
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 5
""").df()

print(df_march.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 2. Fields: feature / label / context / excluded

The fields are divided according to whether they are available before the prediction decision, represent the future outcome, provide context, or contain information that would cause leakage.

### Features

The five features used for the initial feature frame are:

- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `gsc_sum_position`
- `ctr`, calculated from current clicks and impressions

These describe search performance available at the decision moment.

### Label / proxy

- `future_decline` — a binary outcome calculated from later search performance. It represents whether the content item experienced a future decline and is never used as a feature.

### Context

- `report_date`
- `month`
- `client_hash_id`
- `content_hash_id`
- `client_has_gsc`
- `client_has_ga4`
- `gsc_data_available`
- `ga4_data_available`

These fields identify, group, join, or describe the availability of an observation but are not model features.

### Excluded

- Future performance measurements used to construct `future_decline`.
- Any field derived from future performance.

These are excluded because they would leak information that would not be available at the decision moment.

In [51]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df_march = con.execute("""
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 5
""").df()

print(df_march.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 3. Verify it with queries (grain, counts, missing values, windows)



The following queries verify the March row count, date range, data availability, and missing values.

In [52]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.execute("""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [53]:
con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows,
    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_available_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


In [54]:
con.execute("""
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (
        WHERE gsc_impressions IS NULL
    ) AS missing_gsc_impressions,

    COUNT(*) FILTER (
        WHERE gsc_clicks IS NULL
    ) AS missing_gsc_clicks,

    COUNT(*) FILTER (
        WHERE gsc_avg_position IS NULL
    ) AS missing_gsc_avg_position,

    COUNT(*) FILTER (
        WHERE gsc_sum_position IS NULL
    ) AS missing_gsc_sum_position,

    COUNT(*) FILTER (
        WHERE ga4_sessions IS NULL
    ) AS missing_ga4_sessions,

    COUNT(*) FILTER (
        WHERE sessions_organic IS NULL
    ) AS missing_sessions_organic,

    COUNT(*) FILTER (
        WHERE sessions_ai IS NULL
    ) AS missing_sessions_ai,

    COUNT(*) FILTER (
        WHERE scroll_events IS NULL
    ) AS missing_scroll_events

FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,missing_gsc_impressions,missing_gsc_clicks,missing_gsc_avg_position,missing_gsc_sum_position,missing_ga4_sessions,missing_sessions_organic,missing_sessions_ai,missing_scroll_events
0,9841378,0,0,6230317,0,3018741,3018741,3018741,3018741


Five features:

## 5. Initial feature frame

The initial feature frame uses five search-performance signals available at the decision moment:

1. `gsc_impressions`
2. `gsc_clicks`
3. `gsc_avg_position`
4. `gsc_sum_position`
5. `ctr`

`ctr` is calculated from current `gsc_clicks / gsc_impressions` and does not use future information.

In [55]:
features = con.execute("""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,

    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    gsc_sum_position,

    CASE
        WHEN gsc_impressions > 0
        THEN gsc_clicks * 1.0 / gsc_impressions
        ELSE NULL
    END AS ctr

FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)

WHERE gsc_data_available IS TRUE

LIMIT 1000
""").df()

features.head()

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,gsc_sum_position,ctr
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000,67,0.000
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0.000000,0,0.000
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,4.928000,616,0.008
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,4.000000,28,0.000
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,2.272727,25,0.000


### Five-feature frame

The feature frame uses five search-performance features from the March 2026 development window:

1. **gsc_impressions** — available at the decision moment because it records observed search impressions.
2. **gsc_clicks** — available at the decision moment because it records observed search clicks.
3. **gsc_avg_position** — available at the decision moment because it summarizes observed search position.
4. **gsc_sum_position** — available at the decision moment because it summarizes observed search-position measurements.
5. **ctr** — available at the decision moment because it is calculated from observed clicks and impressions.

These features are based only on information available in the development window and are not derived from a future outcome.

In [56]:
con.execute("""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
)
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,first_date,last_date
0,10424730,2026-04-01,2026-04-30


In [57]:
label_df = con.execute("""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.march_impressions,
    a.april_impressions,

    CASE
        WHEN a.april_impressions < m.march_impressions
        THEN 1
        ELSE 0
    END AS future_decline

FROM march m
INNER JOIN april a
    ON m.client_hash_id = a.client_hash_id
    AND m.content_hash_id = a.content_hash_id
""").df()

label_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,march_impressions,april_impressions,future_decline
0,client_62f4a7e64f5e0096,content_143987cfdeaba4c0,345.0,187.0,1
1,client_62f4a7e64f5e0096,content_13a8105125458098,37.0,23.0,1
2,client_62f4a7e64f5e0096,content_e2bd76be7eed690d,15.0,17.0,0
3,client_62f4a7e64f5e0096,content_86ab16840c4e0e1a,854.0,202.0,1
4,client_62f4a7e64f5e0096,content_3f87f49c36774e23,205.0,93.0,1


In [58]:
label_df["future_decline"].value_counts()

,count
future_decline,
1,93779
0,64770


### Leakage experiment

In [59]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Aggregate the March features to one row per content item
feature_df = features.groupby(
    ["client_hash_id", "content_hash_id"],
    as_index=False
).agg({
    "gsc_impressions": "sum",
    "gsc_clicks": "sum",
    "gsc_avg_position": "mean",
    "gsc_sum_position": "sum",
    "ctr": "mean"
})

# Join the future outcome
model_df = feature_df.merge(
    label_df[["client_hash_id", "content_hash_id", "future_decline"]],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "gsc_sum_position",
    "ctr"
]

X = model_df[feature_cols].fillna(0)
y = model_df["future_decline"]

# Honest model
honest_model = DecisionTreeClassifier(
    max_depth=2,
    random_state=42
)
honest_model.fit(X, y)

honest_score = accuracy_score(y, honest_model.predict(X))

# Deliberate leakage: feed the label back in
X_leaky = model_df[feature_cols + ["future_decline"]].fillna(0)

leaky_model = DecisionTreeClassifier(
    max_depth=2,
    random_state=42
)
leaky_model.fit(X_leaky, y)

leaky_score = accuracy_score(y, leaky_model.predict(X_leaky))

print(f"Honest in-sample accuracy: {honest_score:.3f}")
print(f"Leaky in-sample accuracy:  {leaky_score:.3f}")

Honest in-sample accuracy: 0.578
Leaky in-sample accuracy:  1.000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
The data has several important limits. History is unbalanced across clients, so not every client has the same amount of usable search or analytics history. GSC and GA4 availability also differs across observations, meaning GA4-based analysis will cover fewer rows than GSC-based analysis.

For the development window, March 2026 contains 9,841,378 daily rows. GSC data is available for 3,611,061 rows, while GA4 data is available for 413,966 rows. The future-decline label was successfully created for 158,549 observations: 93,779 are labeled as decline and 64,770 as non-decline.

The March-to-April comparison is useful for developing the label, but future performance must never be used as a feature. The final June 2026 month remains sealed for final testing.

Output: The contract defines which observations and pre-decision signals can be used to identify content items that may need refresh review, while keeping future outcome information out of the features.

In [60]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.